# Product access

Testing access to each gridded product (GES DISC Cloud OPeNDAP, IEM Stage IV/MRMS, NCEI CMORPH, CHRS PERSIANN-CCS); IBTrACS.

Cells are the original session code in execution order (exported verbatim; only secrets handling and artifact paths were changed). Each cell starts with a comment giving its original index, conda environment and run status. Exploratory and failed cells are kept, so later cells that rely on their partial state still make sense.


In [ ]:
# Setup: run from the repository root; helper modules live in prod/
import os, sys
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
sys.path.insert(0, 'prod')
os.makedirs('out', exist_ok=True)
from style import apply_figure_style  # stand-in for the figure style used in the session


> Original cell 31 ended with status **error** in the session; it is kept for the record.

In [32]:
# [orig cell 31 | env: precip | status: error]
import requests, os, io, time
class EDSession(requests.Session):
    AUTH_HOST='urs.earthdata.nasa.gov'
    def __init__(s,u,p): super().__init__(); s.auth=(u,p)
    def rebuild_auth(s, prepared_request, response):
        h=prepared_request.headers; url=prepared_request.url
        if 'Authorization' in h:
            o=requests.utils.urlparse(response.request.url).hostname; n=requests.utils.urlparse(url).hostname
            if o!=n and n!=s.AUTH_HOST and o!=s.AUTH_HOST: del h['Authorization']
sess=EDSession(os.environ['EARTHDATA_USER'],os.environ['EARTHDATA_PASS'])
# IMERG grid indices for FL box w/ margin
lon0,lon1=int((-88.6+180)/0.1),int((-79.0+180)/0.1); lat0,lat1=int((23.6+90)/0.1),int((31.9+90)/0.1)
print(lon0,lon1,lat0,lat1)
u=('https://gpm1.gesdisc.eosdis.nasa.gov/opendap/GPM_L3/GPM_3IMERGHH.07/2020/147/'
   '3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5.dap.nc4'
   f'?dap4.ce=/Grid/precipitation[0:1:0][{lon0}:1:{lon1}][{lat0}:1:{lat1}];/Grid/lon[{lon0}:1:{lon1}];/Grid/lat[{lat0}:1:{lat1}]')
t=time.time(); r=sess.get(u,timeout=120); print(r.status_code, len(r.content), round(time.time()-t,2), r.headers.get('content-type'))
open('imerg_test.nc4','wb').write(r.content)
import xarray as xr
try:
    ds=xr.open_dataset('imerg_test.nc4',group='Grid') ; print(ds)
except Exception as e:
    ds=xr.open_dataset('imerg_test.nc4'); print(ds)

914 1010 1135 1219


urllib3.exceptions.ProxyError: ('Unable to connect to proxy', OSError('Tunnel connection failed: 502 Bad Gateway'))

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/requests/adapters.py", line 696, in send
    resp = conn.urlopen(
           ^^^^^^^^^^^^^
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/urllib3/connectionpool.py", line 847, in urlopen
    retries = retries.increment(
              ^^^^^^^^^^^^^^^^^^
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/urllib3/util/retry.py", line 555, in increment
    raise MaxRetryError(_pool, url, reason) from reason  # type: ignore[arg-type]
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
urllib3.exceptions.MaxRetryError: HTTPSConnectionPool(host='gpm1.gesdisc.eosdis.nasa.gov', port=443): Max retries exceeded with url: /opendap/GPM_L3/GPM_3IMERGHH.07/202

In [33]:
%%bash
# [orig cell 32 | env: precip | status: ok]
for u in https://gpm1.gesdisc.eosdis.nasa.gov/opendap/ https://urs.earthdata.nasa.gov/ https://mtarchive.geol.iastate.edu/2020/05/26/stage4/ https://mtarchive.geol.iastate.edu/2020/05/26/mrms/ncep/; do echo "$(curl -s -o /dev/null -w '%{http_code} %{time_total}' --max-time 40 $u) $u"; done

000 15.007145 https://gpm1.gesdisc.eosdis.nasa.gov/opendap/
200 0.381068 https://urs.earthdata.nasa.gov/
404 0.242654 https://mtarchive.geol.iastate.edu/2020/05/26/stage4/
200 0.688507 https://mtarchive.geol.iastate.edu/2020/05/26/mrms/ncep/


In [34]:
%%bash
# [orig cell 33 | env: precip | status: ok]
curl -s -o /dev/null -w '%{http_code} %{time_total}\n' --max-time 90 https://gpm1.gesdisc.eosdis.nasa.gov/ ; curl -sv --max-time 30 https://gpm1.gesdisc.eosdis.nasa.gov/ 2>&1 | grep -iE "HTTP/|proxy|502" | head

000 15.013995
* Uses proxy env variable no_proxy == 'localhost,127.0.0.1,::1,*.local,.local,169.254.0.0/16,10.0.0.0/8,172.16.0.0/12,192.168.0.0/16'
* Uses proxy env variable https_proxy == 'http://localhost:53861'
* CONNECT tunnel: HTTP/1.1 negotiated
* Establish HTTP proxy tunnel to gpm1.gesdisc.eosdis.nasa.gov:443
> CONNECT gpm1.gesdisc.eosdis.nasa.gov:443 HTTP/1.1
> Proxy-Connection: Keep-Alive
< HTTP/1.1 502 Bad Gateway
* CONNECT tunnel failed, response 502


In [35]:
# [orig cell 34 | env: precip | status: ok]
cmr='https://cmr.earthdata.nasa.gov/search/'
r=requests.get(cmr+'collections.json',params={'short_name':'GPM_3IMERGHH','version':'07','provider':'GES_DISC'},timeout=60).json()
for c in r['feed']['entry']: print(c['id'],c['short_name'],c['version_id'],c.get('time_start'))
cid=r['feed']['entry'][0]['id']
g=requests.get(cmr+'granules.json',params={'collection_concept_id':cid,'temporal':'2020-05-26T22:00:00Z,2020-05-26T22:29:00Z','page_size':5},timeout=60).json()
for e in g['feed']['entry']:
    print(e['title']); [print('  ',l['href']) for l in e['links'] if 'opendap' in l['href'].lower() or l['href'].endswith('HDF5')]

C2723754847-GES_DISC GPM_3IMERGHH 07 1998-01-01T00:00:00.000Z
GPM_3IMERGHH.07:3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5
   https://data.gesdisc.earthdata.nasa.gov/data/GPM_L3/GPM_3IMERGHH.07/2020/147/3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5
   s3://gesdisc-cumulus-prod-protected/GPM_L3/GPM_3IMERGHH.07/2020/147/3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5
   https://opendap.earthdata.nasa.gov/collections/C2723754847-GES_DISC/granules/GPM_3IMERGHH.07%3A3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5


In [36]:
# [orig cell 35 | env: precip | status: ok]
sess=EDSession(os.environ['EARTHDATA_USER'],os.environ['EARTHDATA_PASS'])
base_od='https://opendap.earthdata.nasa.gov/collections/C2723754847-GES_DISC/granules/GPM_3IMERGHH.07%3A3B-HHR.MS.MRG.3IMERG.20200526-S220000-E222959.1320.V07B.HDF5'
t=time.time(); r=sess.get(base_od+'.dmr',timeout=120); print(r.status_code, round(time.time()-t,2)); print(r.text[:300] if r.status_code!=200 else [l.strip() for l in r.text.splitlines() if 'precipitation' in l or 'Dim name' in l][:12])
ce=f'/Grid/precipitation[0:1:0][{lon0}:1:{lon1}][{lat0}:1:{lat1}];/Grid/lon[{lon0}:1:{lon1}];/Grid/lat[{lat0}:1:{lat1}];/Grid/time'
t=time.time(); r=sess.get(base_od+'.dap.nc4',params={'dap4.ce':ce},timeout=120); print(r.status_code,len(r.content),round(time.time()-t,2), r.headers.get('content-type'))
open('imerg_test.nc4','wb').write(r.content)

200 2.99
['<Dim name="/Grid/lat"/>', '<Dim name="/Grid/lon"/>', '<Dim name="/Grid/time"/>', '<Dim name="/Grid/lat"/>', '<Dim name="/Grid/latv"/>', '<Float32 name="precipitation">', '<Dim name="/Grid/time"/>', '<Dim name="/Grid/lon"/>', '<Dim name="/Grid/lat"/>', 'precipitation estimate;  formerly precipitationCal', '<Float32 name="precipitationQualityIndex">', '<Dim name="/Grid/time"/>']
200 38914 1.51 application/x-netcdf;ver=4


In [37]:
# [orig cell 36 | env: precip | status: ok]
import xarray as xr, netCDF4
nc=netCDF4.Dataset('imerg_test.nc4'); print(nc.groups.keys() if nc.groups else list(nc.variables))
ds=xr.open_dataset('imerg_test.nc4',group='Grid') if nc.groups else xr.open_dataset('imerg_test.nc4')
print(ds.precipitation.dims, ds.precipitation.shape, float(ds.lon.min()),float(ds.lon.max()),float(ds.lat.min()),float(ds.lat.max()), ds.precipitation.attrs.get('units'))
print(ds.precipitation.sel(lon=-80.3,lat=25.8,method='nearest').values, float(ds.precipitation.max()))

dict_keys(['Grid'])
('time', 'lon', 'lat') (1, 97, 85) -88.54999542236328 -78.94999694824219 23.549999237060547 31.94999885559082 mm/hr
[9.87] 31.489999771118164


In [38]:
# [orig cell 37 | env: precip | status: ok]
for d in ['2015-01-01T00:00:00Z,2015-01-01T00:29:00Z','2019-07-01T12:00:00Z,2019-07-01T12:29:00Z','2024-08-01T12:00:00Z,2024-08-01T12:29:00Z','2025-09-15T12:00:00Z,2025-09-15T12:29:00Z','2025-12-30T12:00:00Z,2025-12-30T12:29:00Z']:
    g=requests.get(cmr+'granules.json',params={'collection_concept_id':cid,'temporal':d,'page_size':5},timeout=60).json()['feed']['entry']
    print(d[:10],[e['title'] for e in g])
print(e20.hour_start_utc.max())

2015-01-01 ['GPM_3IMERGHH.07:3B-HHR.MS.MRG.3IMERG.20150101-S000000-E002959.0000.V07B.HDF5']
2019-07-01 ['GPM_3IMERGHH.07:3B-HHR.MS.MRG.3IMERG.20190701-S120000-E122959.0720.V07B.HDF5']
2024-08-01 ['GPM_3IMERGHH.07:3B-HHR.MS.MRG.3IMERG.20240801-S120000-E122959.0720.V07B.HDF5']
2025-09-15 ['GPM_3IMERGHH.07:3B-HHR.MS.MRG.3IMERG.20250915-S120000-E122959.0720.V07B.HDF5']
2025-12-30 []
2025-12-12 16:00:00


In [39]:
%%bash
# [orig cell 38 | env: precip | status: ok]
B=https://mtarchive.geol.iastate.edu
curl -s $B/2020/05/26/ | grep -o 'href="[^"]*"' | head -30 | tr '\n' ' '; echo
curl -s $B/2020/05/26/mrms/ncep/ | grep -o 'href="[^"]*"' | grep -i -E "qpe|precip" | tr '\n' ' '; echo
curl -s $B/2016/05/17/mrms/ncep/ | grep -o 'href="[^"]*"' | grep -i -E "qpe|precip|gauge" | tr '\n' ' '; echo

href="?C=N;O=D" href="?C=M;O=A" href="?C=S;O=A" href="?C=D;O=A" href="/2020/05/" href="bufkit/" href="cod/" href="gempak/" href="grib2/" href="mrms/" href="pix/" href="text/" 
href="GaugeCorr_QPE_01H/" href="GaugeCorr_QPE_24H/" href="GaugeCorr_QPE_72H/" href="MountainMapper_QPE_01H/" href="MountainMapper_QPE_03H/" href="MountainMapper_QPE_06H/" href="MountainMapper_QPE_12H/" href="MountainMapper_QPE_24H/" href="MountainMapper_QPE_48H/" href="MountainMapper_QPE_72H/" href="PrecipFlag/" href="PrecipRate/" href="RadarOnly_QPE_01H/" href="RadarOnly_QPE_24H/" href="RadarOnly_QPE_72H/" 
href="GaugeCorr_QPE_01H/" href="PrecipFlag/" href="PrecipRate/" href="RadarOnly_QPE_01H/" href="RadarOnly_QPE_24H/" 


In [40]:
%%bash
# [orig cell 39 | env: precip | status: ok]
B=https://mtarchive.geol.iastate.edu
curl -s $B/2020/05/26/grib2/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo
curl -s $B/2020/05/26/stage4/ -o /dev/null -w '%{http_code}\n'
curl -s $B/2020/05/26/mrms/ncep/GaugeCorr_QPE_01H/ | grep -o 'href="[^"]*"' | sed -n '5,8p'
curl -s $B/2023/08/01/mrms/ncep/ | grep -o 'href="[^"]*"' | grep -i -E "qpe_01h|multisensor" | tr '\n' ' '; echo
curl -s $B/2023/08/01/mrms/ncep/MultiSensor_QPE_01H_Pass2/ | grep -o 'href="[^"]*"' | sed -n '5,7p'
curl -s $B/2015/03/01/mrms/ncep/ | grep -o 'href="[^"]*"' | grep -i qpe | tr '\n' ' '

href="?C=N;O=D" href="?C=M;O=A" href="?C=S;O=A" href="?C=D;O=A" href="/2020/05/26/" href="ncep/" 
404
href="/2020/05/26/mrms/ncep/"
href="GaugeCorr_QPE_01H_00.00_20200526-000000.grib2.gz"
href="GaugeCorr_QPE_01H_00.00_20200526-010000.grib2.gz"
href="GaugeCorr_QPE_01H_00.00_20200526-020000.grib2.gz"
href="MultiSensor_QPE_01H_Pass2/" href="MultiSensor_QPE_24H_Pass2/" href="MultiSensor_QPE_72H_Pass2/" href="RadarOnly_QPE_01H/" 
href="/2023/08/01/mrms/ncep/"
href="MultiSensor_QPE_01H_Pass2_00.00_20230801-000000.grib2.gz"
href="MultiSensor_QPE_01H_Pass2_00.00_20230801-010000.grib2.gz"
href="RadarOnly_QPE_01H/" href="RadarOnly_QPE_24H/" 

In [41]:
%%bash
# [orig cell 40 | env: precip | status: ok]
B=https://mtarchive.geol.iastate.edu
curl -s $B/2020/05/26/grib2/ncep/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo
curl -s $B/2020/05/26/grib2/ncep/STAGE4/ | grep -o 'href="[^"]*"' | sed -n '5,9p'
curl -sI $B/2020/05/26/mrms/ncep/GaugeCorr_QPE_01H/GaugeCorr_QPE_01H_00.00_20200526-230000.grib2.gz | grep -i length

href="?C=N;O=D" href="?C=M;O=A" href="?C=S;O=A" href="?C=D;O=A" href="/2020/05/26/grib2/" href="NAM218/" href="NBM/" href="RTMA/" 
Content-Length: 885584


In [42]:
%%bash
# [orig cell 41 | env: precip | status: ok]
B=https://mesonet.agron.iastate.edu/archive/data
curl -s $B/2020/05/26/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo
curl -s $B/2020/05/26/stage4/ | grep -o 'href="[^"]*"' | grep 01h | head -4
curl -s $B/2016/05/17/stage4/ | grep -o 'href="[^"]*"' | grep 01h | head -2
curl -s $B/2025/08/01/stage4/ | grep -o 'href="[^"]*"' | grep 01h | head -2

href="../" href="awos_rtp.shef" href="awos_rtp_00z.shef" href="camera/" href="comprad/" href="conus_asos_12z_low.png" href="conus_gfs_mos_T_bias_00.png" href="conus_gfs_mos_T_bias_03.png" href="conus_gfs_mos_T_bias_06.png" href="conus_gfs_mos_T_bias_09.png" href="conus_gfs_mos_T_bias_12.png" href="conus_gfs_mos_T_bias_15.png" href="conus_gfs_mos_T_bias_18.png" href="conus_gfs_mos_T_bias_21.png" href="conus_nam_mos_T_bias_00.png" href="conus_nam_mos_T_bias_03.png" href="conus_nam_mos_T_bias_06.png" href="conus_nam_mos_T_bias_09.png" href="conus_nam_mos_T_bias_12.png" href="conus_nam_mos_T_bias_15.png" href="conus_nam_mos_T_bias_18.png" href="conus_nam_mos_T_bias_21.png" href="conus_stage4_1d.png" href="conus_stage4_1h_00.png" href="conus_stage4_1h_01.png" href="conus_stage4_1h_02.png" href="conus_stage4_1h_03.png" href="conus_stage4_1h_04.png" href="conus_stage4_1h_05.png" href="conus_stage4_1h_06.png" href="conus_stage4_1h_07.png" href="conus_stage4_1h_08.png" href="conus_stage4_1h_09.

In [43]:
%%bash
# [orig cell 42 | env: precip | status: ok]
B=https://mesonet.agron.iastate.edu/archive/data
for d in 2016/05/17 2020/05/26 2025/08/01; do echo "== $d"; curl -s $B/$d/stage4/ | grep -o 'href="[^"]*"' | grep -E '01h' | head -3 | tr '\n' ' '; echo; done
curl -sI $B/2020/05/26/stage4/ST4.2020052622.01h.grib | grep -iE "HTTP|length"

== 2016/05/17
href="ST2ml.2016051700.01h.grib" href="ST2ml.2016051701.01h.grib" href="ST2ml.2016051702.01h.grib" 
== 2020/05/26
href="ST2ml.2020052600.01h.grib" href="ST2ml.2020052601.01h.grib" href="ST2ml.2020052602.01h.grib" 
== 2025/08/01
href="ST4.2025080100.01h.grib" href="ST4.2025080101.01h.grib" href="ST4.2025080102.01h.grib" 
HTTP/1.1 200 Connection Established
HTTP/1.1 200 OK
Content-Length: 1083196


In [44]:
# [orig cell 43 | env: precip | status: ok]
import gzip, eccodes, tempfile
def grib_from_bytes(b):
    gid=eccodes.codes_new_from_message(b)
    try:
        vals=eccodes.codes_get_values(gid); ni=eccodes.codes_get(gid,'Ni'); nj=eccodes.codes_get(gid,'Nj')
        meta={k:eccodes.codes_get(gid,k) for k in ['gridType','shortName','missingValue']}
        for k in ['latitudeOfFirstGridPointInDegrees','longitudeOfFirstGridPointInDegrees','latitudeOfLastGridPointInDegrees','longitudeOfLastGridPointInDegrees','iDirectionIncrementInDegrees','jDirectionIncrementInDegrees']:
            try: meta[k]=eccodes.codes_get(gid,k)
            except Exception: pass
        if meta['gridType']!='regular_ll':
            lats=eccodes.codes_get_array(gid,'latitudes').reshape(nj,ni); lons=eccodes.codes_get_array(gid,'longitudes').reshape(nj,ni)
        else: lats=lons=None
        bm=eccodes.codes_get(gid,'bitmapPresent')
        return vals.reshape(nj,ni),meta,lats,lons
    finally: eccodes.codes_release(gid)
r=requests.get('https://mesonet.agron.iastate.edu/archive/data/2020/05/26/stage4/ST4.2020052623.01h.grib',timeout=60)
v4,m4,la4,lo4=grib_from_bytes(r.content); print(v4.shape,m4); 
lo4=np.where(lo4>180,lo4-360,lo4)
d=(la4-25.788)**2+((lo4+80.317)*np.cos(np.radians(25.8)))**2; i,j=np.unravel_index(d.argmin(),d.shape); print('ST4 Miami cell',i,j,v4[i,j], v4[i-2:i+3,j-2:j+3].round(1))
r=requests.get('https://mtarchive.geol.iastate.edu/2020/05/26/mrms/ncep/GaugeCorr_QPE_01H/GaugeCorr_QPE_01H_00.00_20200526-230000.grib2.gz',timeout=60)
vm,mm,_,_=grib_from_bytes(gzip.decompress(r.content)); print(vm.shape,mm)

(881, 1121) {'gridType': 'polar_stereographic', 'shortName': 'tp', 'missingValue': 9999, 'latitudeOfFirstGridPointInDegrees': 23.117, 'longitudeOfFirstGridPointInDegrees': -119.023}
ST4 Miami cell 176 1053 54.75 [[33.2 40.6 57.4 33.4 24.4]
 [38.8 41.4 61.  51.2 48. ]
 [32.2 36.2 54.8 56.9 55.5]
 [27.5 30.8 51.8 77.2 66.2]
 [27.2 28.6 38.4 47.9 48.4]]
(3500, 7000) {'gridType': 'regular_ll', 'shortName': 'unknown', 'missingValue': 9999, 'latitudeOfFirstGridPointInDegrees': 54.995, 'longitudeOfFirstGridPointInDegrees': 230.005, 'latitudeOfLastGridPointInDegrees': 20.005001, 'longitudeOfLastGridPointInDegrees': 299.994998, 'iDirectionIncrementInDegrees': 0.01, 'jDirectionIncrementInDegrees': 0.01}


In [45]:
%%bash
# [orig cell 44 | env: precip | status: ok]
mkdir -p prod && cat > prod/common.py <<'EOF'
import numpy as np, pandas as pd, json, os
W=8; OFF=3   # window rows i0-3..i0+4
def stations():
    V=pd.read_csv('out/valid_stations.csv'); return V[['id','lat','lon']].reset_index(drop=True)
def hours():
    return pd.read_parquet('out/needed_hours.parquet').hour_start_utc.reset_index(drop=True)
def _dist(lat1,lon1,lat2,lon2):
    p=np.pi/180; a=np.sin((lat2-lat1)*p/2)**2+np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2
    return 6371*2*np.arcsin(np.sqrt(a))
def window_index(lat2d, lon2d):
    """lat2d/lon2d: cell-centre arrays of the (sub)grid. Returns per-station dict with window origin and signs."""
    S=stations(); out=[]
    ny,nx=lat2d.shape
    for _,s in S.iterrows():
        d=_dist(s.lat,s.lon,lat2d,lon2d); i0,j0=np.unravel_index(np.nanargmin(d),d.shape)
        def dd(i,j): return d[i,j] if 0<=i<ny and 0<=j<nx else np.inf
        si=1 if dd(i0+1,j0)<dd(i0-1,j0) else -1; sj=1 if dd(i0,j0+1)<dd(i0,j0-1) else -1
        r0,c0=i0-OFF,j0-OFF
        ok = r0>=0 and c0>=0 and r0+W<=ny and c0+W<=nx
        # pixel size km (mean spacing)
        dy=_dist(lat2d[i0,j0],lon2d[i0,j0],lat2d[min(i0+1,ny-1),j0],lon2d[min(i0+1,ny-1),j0])
        dx=_dist(lat2d[i0,j0],lon2d[i0,j0],lat2d[i0,min(j0+1,nx-1)],lon2d[i0,min(j0+1,nx-1)])
        out.append(dict(id=s.id,i0=int(i0),j0=int(j0),r0=int(r0),c0=int(c0),si=si,sj=sj,ok=bool(ok),dist_km=float(d[i0,j0]),dx_km=float(dx),dy_km=float(dy),
                        win_lat=lat2d[max(r0,0):r0+W,max(c0,0):c0+W].tolist(), win_lon=lon2d[max(r0,0):r0+W,max(c0,0):c0+W].tolist()))
    return out
def cut(field, widx):
    """field 2D on same grid -> array (nstations,W,W)"""
    o=np.full((len(widx),W,W),np.nan,np.float32)
    for k,w in enumerate(widx):
        if w['ok']: o[k]=field[w['r0']:w['r0']+W, w['c0']:w['c0']+W]
    return o
def neighborhood_masks(w):
    """boolean masks (W,W) for N1,N4,N9,N16 in window coords"""
    i,j=OFF,OFF; si,sj=w['si'],w['sj']; m={}
    def blk(rows,cols):
        a=np.zeros((W,W),bool); a[np.ix_(rows,cols)]=True; return a
    m['N1']=blk([i],[j])
    m['N4']=blk(sorted([i,i+si]),sorted([j,j+sj]))
    m['N9']=blk([i-1,i,i+1],[j-1,j,j+1])
    m['N16']=blk(sorted([i-si,i,i+si,i+2*si]),sorted([j-sj,j,j+sj,j+2*sj]))
    return m
def save(name, arr, widx, hrs, extra=None):
    np.save(f'prod/{name}_win.npy', arr)
    meta=dict(name=name,hours=[str(h) for h in hrs],stations=[w['id'] for w in widx],widx=widx,extra=extra or {})
    json.dump(meta,open(f'prod/{name}_meta.json','w'))
EOF
echo ok

ok


In [46]:
%%bash
# [orig cell 45 | env: precip | status: ok]
cat > prod/hourly_grib.py <<'EOF'
import sys, os, gzip, time, json, numpy as np, pandas as pd, requests, eccodes, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
prod=sys.argv[1]
H=hours(); n=len(H)
def url(h):
    t=h+pd.Timedelta(hours=1)   # files are stamped at hour end
    if prod=='stage4':
        return [f'https://mesonet.agron.iastate.edu/archive/data/{t:%Y/%m/%d}/stage4/ST4.{t:%Y%m%d%H}.01h.grib']
    if prod=='mrms_radaronly':
        return [f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/RadarOnly_QPE_01H/RadarOnly_QPE_01H_00.00_{t:%Y%m%d-%H}0000.grib2.gz']
    if prod=='mrms_gaugecorr':
        return [f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/MultiSensor_QPE_01H_Pass2/MultiSensor_QPE_01H_Pass2_00.00_{t:%Y%m%d-%H}0000.grib2.gz',
                f'https://mtarchive.geol.iastate.edu/{t:%Y/%m/%d}/mrms/ncep/GaugeCorr_QPE_01H/GaugeCorr_QPE_01H_00.00_{t:%Y%m%d-%H}0000.grib2.gz']
def decode(b, want_ll=False):
    if b[:2]==b'\x1f\x8b': b=gzip.decompress(b)
    gid=eccodes.codes_new_from_message(b)
    try:
        ni=eccodes.codes_get(gid,'Ni'); nj=eccodes.codes_get(gid,'Nj'); v=eccodes.codes_get_values(gid).reshape(nj,ni).astype(np.float32)
        mv=eccodes.codes_get(gid,'missingValue'); v[v==mv]=np.nan
        ll=None
        if want_ll:
            la=eccodes.codes_get_array(gid,'latitudes').reshape(nj,ni); lo=eccodes.codes_get_array(gid,'longitudes').reshape(nj,ni); lo=np.where(lo>180,lo-360,lo); ll=(la,lo)
        return v,ll
    finally: eccodes.codes_release(gid)
def fetch(h):
    for u in url(h):
        for a in range(3):
            try:
                r=requests.get(u,timeout=120)
                if r.status_code==404: break
                if r.ok: return r.content, u
            except Exception: time.sleep(3)
    return None, None
# grid from a reference hour
ref=None
for h in H[len(H)//2:]:
    b,u=fetch(h)
    if b: ref=decode(b,True); break
v,(la,lo)=ref
if prod.startswith('mrms'): v[v<0]=np.nan  # -3 = no coverage
widx=window_index(la,lo); print(prod,'grid',v.shape,'stations ok',sum(w['ok'] for w in widx),flush=True)
del la,lo
WIDX=widx
def work(k):
    b,u=fetch(H[k])
    if b is None: return k,None,None
    try:
        f,_=decode(b)
        if prod.startswith('mrms'): f[f<0]=np.nan
        return k,cut(f,WIDX),u.split('/')[-1][:20]
    except Exception as e: return k,None,'err'
out=np.full((n,len(widx),W,W),np.nan,np.float32); src=['']*n
t0=time.time(); done=0
with cf.ThreadPoolExecutor(12) as ex:
    for k,arr,s in ex.map(work,range(n)):
        done+=1
        if arr is not None: out[k]=arr; src[k]=s
        if done%2000==0: print(done,round((time.time()-t0)/60,1),'min',flush=True)
save(prod,out,widx,H,extra=dict(source=src))
print(prod,'done; hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',n, round((time.time()-t0)/60,1),'min')
EOF
echo ok

ok


In [47]:
%%bash
# [orig cell 46 | env: precip | status: ok]
ls prod/imerg | wc -l
python - <<'EOF'
import os,re
p=os.environ.get('CDS_PASS',''); e=os.environ.get('CDS_EMAIL','')
print('pass looks like UUID token:', bool(re.fullmatch(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}',p.strip())), 'len',len(p), '| email field has @:', '@' in e, '| email looks like UUID:', bool(re.fullmatch(r'[0-9a-f-]{36}',e.strip())))
EOF

    4595
pass looks like UUID token: False len 15 | email field has @: True | email looks like UUID: False


In [48]:
# [orig cell 47 | env: precip | status: ok]
for sn,ver in [('M2T1NXFLX','5.12.4'),('NLDAS_FORA0125_H','2.0')]:
    r=requests.get(cmr+'collections.json',params={'short_name':sn,'version':ver},timeout=60).json()['feed']['entry']
    for c in r:
        print(c['id'],c['short_name'],c['version_id'])
        g=requests.get(cmr+'granules.json',params={'collection_concept_id':c['id'],'temporal':'2020-05-26T00:00:00Z,2020-05-26T01:00:00Z','page_size':3},timeout=60).json()['feed']['entry']
        for e in g[:2]:
            print('  ',e['title']); [print('     ',l['href']) for l in e['links'] if 'opendap.earthdata' in l['href']]

C1276812838-GES_DISC M2T1NXFLX 5.12.4
   M2T1NXFLX.5.12.4:MERRA2_400.tavg1_2d_flx_Nx.20200526.nc4
      https://opendap.earthdata.nasa.gov/collections/C1276812838-GES_DISC/granules/M2T1NXFLX.5.12.4%3AMERRA2_400.tavg1_2d_flx_Nx.20200526.nc4
C2033151148-GES_DISC NLDAS_FORA0125_H 2.0
   NLDAS_FORA0125_H.2.0:NLDAS_FORA0125_H.A20200526.0000.020.nc
      https://opendap.earthdata.nasa.gov/collections/C2033151148-GES_DISC/granules/NLDAS_FORA0125_H.2.0%3ANLDAS_FORA0125_H.A20200526.0000.020.nc
   NLDAS_FORA0125_H.2.0:NLDAS_FORA0125_H.A20200526.0100.020.nc
      https://opendap.earthdata.nasa.gov/collections/C2033151148-GES_DISC/granules/NLDAS_FORA0125_H.2.0%3ANLDAS_FORA0125_H.A20200526.0100.020.nc


In [49]:
# [orig cell 48 | env: precip | status: ok]
sess=EDSession(os.environ['EARTHDATA_USER'],os.environ['EARTHDATA_PASS'])
mb='https://opendap.earthdata.nasa.gov/collections/C1276812838-GES_DISC/granules/M2T1NXFLX.5.12.4%3AMERRA2_400.tavg1_2d_flx_Nx.20200526.nc4'
r=sess.get(mb+'.dap.nc4',params={'dap4.ce':'/PRECTOTCORR[0:1:23][226:1:246][143:1:163];/lat[226:1:246];/lon[143:1:163];/time'},timeout=120); print(r.status_code,len(r.content))
open('m2_test.nc4','wb').write(r.content); m=xr.open_dataset('m2_test.nc4'); print(m.PRECTOTCORR.dims, m.lat.values[[0,-1]], m.lon.values[[0,-1]], m.time.values[:2])
print((m.PRECTOTCORR.sel(lat=25.8,lon=-80.3,method='nearest')*3600).values.round(1))
nb='https://opendap.earthdata.nasa.gov/collections/C2033151148-GES_DISC/granules/NLDAS_FORA0125_H.2.0%3ANLDAS_FORA0125_H.A20200526.2200.020.nc'
r=sess.get(nb+'.dmr',timeout=120); print([l.strip()[:80] for l in r.text.splitlines() if 'name="Rainf"' in l or 'Dim name' in l][:8])
r=sess.get(nb+'.dap.nc4',params={'dap4.ce':'/Rainf[0:1:0][0:1:62][286:1:352];/lat[0:1:62];/lon[286:1:352];/time'},timeout=120); print(r.status_code,len(r.content))
open('nl_test.nc4','wb').write(r.content); nl=xr.open_dataset('nl_test.nc4'); print(nl.Rainf.dims, nl.lat.values[[0,-1]], nl.lon.values[[0,-1]], nl.time.values, nl.Rainf.attrs.get('units'))
print(nl.Rainf.sel(lat=25.8,lon=-80.3,method='nearest').values)

200 62718
('time', 'lat', 'lon') [23. 33.] [-90.625 -78.125] ['2020-05-26T00:30:00.000000000' '2020-05-26T01:30:00.000000000']
[2.9 2.2 1.3 0.6 0.5 0.4 0.6 0.6 0.5 0.4 0.4 0.3 0.4 0.5 0.4 0.3 0.3 0.3
 0.3 0.4 0.4 0.4 0.4 0.5]
['<Dim name="/time"/>', '<Dim name="/lon"/>', '<Dim name="/lat"/>', '<Dim name="/time"/>', '<Dim name="/lat"/>', '<Dim name="/lon"/>', '<Dim name="/time"/>', '<Dim name="/lat"/>']
200 36295
('time', 'lat', 'lon') [25.0625 32.8125] [-89.1875 -80.9375] ['2020-05-26T22:00:00.000000000'] kg m-2
[0.]


In [50]:
%%bash
# [orig cell 49 | env: precip | status: ok]
cat > prod/nasa_od.py <<'EOF'
import sys, os, io, time, numpy as np, pandas as pd, requests, xarray as xr, threading, concurrent.futures as cf
sys.path.insert(0,'prod'); from common import *
prod=sys.argv[1]
class EDSession(requests.Session):
    def __init__(s,u,p): super().__init__(); s.auth=(u,p)
    def rebuild_auth(s, pr, resp):
        h=pr.headers
        if 'Authorization' in h:
            o=requests.utils.urlparse(resp.request.url).hostname; n=requests.utils.urlparse(pr.url).hostname
            if o!=n and 'urs.earthdata' not in (n or '') and 'urs.earthdata' not in (o or ''): del h['Authorization']
loc=threading.local()
def S():
    if not hasattr(loc,'s'): loc.s=EDSession(os.environ['EARTHDATA_USER'],os.environ['EARTHDATA_PASS'])
    return loc.s
H=hours(); n=len(H); hidx={h:k for k,h in enumerate(H)}
cmr='https://cmr.earthdata.nasa.gov/search/granules.json'
if prod=='merra2':
    cid='C1276812838-GES_DISC'; ce='/PRECTOTCORR[0:1:23][226:1:246][143:1:163]'; lat=np.arange(226,247)*0.5-90; lon=np.arange(143,164)*0.625-180
    # granule titles per day (stream 400/401)
    titles={}; hdr={}
    params={'collection_concept_id':cid,'temporal':'2015-01-01T00:00:00Z,2025-12-31T23:59:59Z','page_size':2000}
    while True:
        r=requests.get(cmr,params=params,headers=hdr,timeout=120); ent=r.json()['feed']['entry']
        for e in ent: titles[e['title'].split(':')[1].split('.')[-2]]=e['title']
        sa=r.headers.get('CMR-Search-After')
        if not ent or not sa: break
        hdr={'CMR-Search-After':sa}
    units=pd.Series(H).dt.floor('D').unique()
    jobs=[(d, titles.get(f'{d:%Y%m%d}')) for d in units]
    print('merra days',len(jobs),'with granule',sum(t is not None for _,t in jobs),flush=True)
elif prod=='nldas2':
    cid='C2033151148-GES_DISC'; ce='/Rainf[0:1:0][0:1:62][286:1:370]'; lat=25.0625+np.arange(0,63)*0.125; lon=-124.9375+np.arange(286,371)*0.125
    jobs=[(h, f'NLDAS_FORA0125_H.2.0:NLDAS_FORA0125_H.A{h:%Y%m%d.%H}00.020.nc') for h in H]
la2,lo2=np.meshgrid(lat,lon,indexing='ij'); widx=window_index(la2,lo2); print('stations ok',sum(w['ok'] for w in widx),flush=True)
base=f'https://opendap.earthdata.nasa.gov/collections/{cid}/granules/'
def get(job):
    key,title=job
    if title is None: return key,None
    for a in range(4):
        try:
            r=S().get(base+title.replace(':','%3A')+'.dap.nc4',params={'dap4.ce':ce},timeout=180)
            if r.status_code==404: return key,None
            if r.ok:
                ds=xr.open_dataset(io.BytesIO(r.content),engine='h5netcdf')
                v=ds['PRECTOTCORR'].values*3600 if prod=='merra2' else ds['Rainf'].values   # -> mm per hour
                return key,v.astype(np.float32)
        except Exception as e: pass
        time.sleep(3*(a+1))
    return key,'fail'
out=np.full((n,len(widx),W,W),np.nan,np.float32); t0=time.time(); nf=0
with cf.ThreadPoolExecutor(12) as ex:
    for i,(key,v) in enumerate(ex.map(get,jobs)):
        if v is None or isinstance(v,str): nf+= isinstance(v,str); continue
        if prod=='merra2':
            for hh in range(24):   # MERRA-2 hour index hh = average over [hh:00, hh+1:00)
                k=hidx.get(key+pd.Timedelta(hours=hh))
                if k is not None: out[k]=cut(v[hh],widx)
        else:
            out[hidx[key]]=cut(v[0],widx)
        if (i+1)%2000==0: print(i+1,round((time.time()-t0)/60,1),'min',flush=True)
save(prod,out,widx,H)
print(prod,'done; fails',nf,'hours with data',int(np.isfinite(out[:,:,OFF,OFF]).any(1).sum()),'of',n,round((time.time()-t0)/60,1),'min')
EOF
echo ok

ok


In [51]:
%%bash
# [orig cell 50 | env: precip | status: ok]
B=https://www.ncei.noaa.gov/data/cmorph-high-resolution-global-precipitation-estimates/access
curl -s $B/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo
curl -s $B/30min/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo
curl -s $B/30min/8km/ | grep -o 'href="[^"]*"' | tail -5 | tr '\n' ' '; echo
curl -s $B/30min/8km/2020/05/26/ | grep -o 'href="[^"]*"' | sed -n '1,4p' | tr '\n' ' '; echo
curl -s $B/30min/8km/2025/ | grep -o 'href="[^"]*"' | tail -3 | tr '\n' ' '

href=".." href="30min" href="daily" href="hourly" 
href=".." href="8km" 
href="2022" href="2023" href="2024" href="2025" href="2026" 
href=".." href="CMORPH_V1.0_ADJ_8km-30min_2020052600.nc" href="CMORPH_V1.0_ADJ_8km-30min_2020052601.nc" href="CMORPH_V1.0_ADJ_8km-30min_2020052602.nc" 
href="10" href="11" href="12" 

In [52]:
%%bash
# [orig cell 51 | env: precip | status: ok]
B=https://www.ncei.noaa.gov/data/cmorph-high-resolution-global-precipitation-estimates/access
curl -sI $B/30min/8km/2020/05/26/CMORPH_V1.0_ADJ_8km-30min_2020052622.nc | grep -i length
curl -s $B/hourly/ | grep -o 'href="[^"]*"' | tr '\n' ' '; echo; curl -s $B/hourly/0.25deg/2020/05/26/ 2>/dev/null| grep -o 'href="[^"]*"' | sed -n 2,3p
curl -s $B/30min/8km/2025/12/31/ | grep -o 'href="[^"]*"' | sed -n 2,3p; curl -s $B/30min/8km/2015/01/01/ | grep -o 'href="[^"]*"' | sed -n 2,3p
curl -s -o cm_test.nc $B/30min/8km/2020/05/26/CMORPH_V1.0_ADJ_8km-30min_2020052622.nc; ls -la cm_test.nc

Content-Length: 1639771
href=".." href="0.25deg" 
href="CMORPH_V1.0_ADJ_0.25deg-HLY_2020052600.nc"
href="CMORPH_V1.0_ADJ_0.25deg-HLY_2020052601.nc"
href="CMORPH_V1.0_ADJ_8km-30min_2025123100.nc"
href="CMORPH_V1.0_ADJ_8km-30min_2025123101.nc"
href="CMORPH_V1.0_ADJ_8km-30min_2015010100.nc"
href="CMORPH_V1.0_ADJ_8km-30min_2015010101.nc"
-rw-r--r--@ 1 mehdihatamigoloujeh  staff  1639771 Oct 18 21:01 cm_test.nc


In [53]:
# [orig cell 52 | env: precip | status: ok]
c=xr.open_dataset('cm_test.nc'); print(c); 
v=c.cmorph if 'cmorph' in c else c[list(c.data_vars)[0]]
print(v.sel(lat=25.8,lon=360-80.3 if float(c.lon.max())>180 else -80.3,method='nearest').values)

<xarray.Dataset> Size: 65MB
Dimensions:      (time: 2, nv: 2, lat: 1649, lon: 4948)
Coordinates:
  * time         (time) datetime64[ns] 16B 2020-05-26T22:00:00 2020-05-26T22:...
  * lat          (lat) float64 13kB -59.96 -59.89 -59.82 ... 59.82 59.89 59.96
  * lon          (lon) float64 40kB 0.03638 0.1091 0.1819 ... 359.8 359.9 360.0
Dimensions without coordinates: nv
Data variables:
    time_bounds  (time, nv) datetime64[ns] 32B ...
    lat_bounds   (lat, nv) float64 26kB ...
    lon_bounds   (lon, nv) float64 79kB ...
    cmorph       (time, lat, lon) float32 65MB ...
Attributes: (12/57)
    ncei_template_version:      NCEI_NetCDF_Grid_template_V2.0
    title:                      NOAA Climate Data Record (CDR) of CPC Morphin...
    keywords:                   Precipitation, Satellite, High-Resolution, Gl...
    summary:                    The CMORPH CDR is a reprocessed and bias-corr...
    references:                 Xie, P., et al. (2017), Reprocessed, Bias-Cor...
    Conventions

In [54]:
%%bash
# [orig cell 53 | env: precip | status: ok]
B=https://persiann.eng.uci.edu/CHRSdata
curl -s --max-time 30 $B/ | grep -o 'href="[^"]*"' | head -20 | tr '\n' ' '; echo
curl -s --max-time 30 $B/PERSIANN-CCS/ | grep -o 'href="[^"]*"' | head -20 | tr '\n' ' '; echo
curl -s --max-time 30 $B/PERSIANN-CCS/hrly/ | grep -o 'href="[^"]*"' | tail -5 | tr '\n' ' '

href="?C=N;O=D" href="?C=M;O=A" href="?C=S;O=A" href="?C=D;O=A" href="/" href="30PDIR/" href="PCCSCDR/" href="PCCSCDR_B1/" href="PCCSCDR_CPC/" href="PDIRNow/" href="PERSIANN-CCS/" href="PERSIANN-CDR/" href="PERSIANN/" href="PUnet/" href="PUnetCDR/" href="dtb/" href="ksa_sample_input/" href="vesta/" 
href="?C=N;O=D" href="?C=M;O=A" href="?C=S;O=A" href="?C=D;O=A" href="/CHRSdata/" href="3hrly/" href="6hrly/" href="daily/" href="hrly/" href="mthly/" href="yearly/" 


In [55]:
%%bash
# [orig cell 54 | env: precip | status: ok]
B=https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly
curl -s --max-time 60 $B/ -o pccs_list.html; wc -c pccs_list.html; grep -o 'href="[^"]*"' pccs_list.html | sed -n '5,10p' | tr '\n' ' '; echo; grep -o 'href="[^"]*"' pccs_list.html | tail -4 | tr '\n' ' '

    6565 pccs_list.html
href="/CHRSdata/PERSIANN-CCS/" href="2000/" href="2001/" href="2002/" href="2003/" href="2004/" 
href="2024/" href="2025/" href="2026/" href="readme.hrly" 

In [56]:
%%bash
# [orig cell 55 | env: precip | status: ok]
B=https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly
curl -s --max-time 60 $B/readme.hrly | head -60
curl -s --max-time 60 $B/2020/ | grep -o 'href="[^"]*"' | sed -n '5,8p' | tr '\n' ' '

hrly global ccs data is in row centric format.

name scheme:
	rgccs1h0934923.bin.gz
w/ YYDDDHH  date/time 
The HH represents the beginning of the hrly time period like so:
	HH:00 to HH:59

data are organized into YYYY subdirectories.

Format is:
    binary in row centric format.
    short integer
    data are big-endian.
    units mm/hr *100

coverage is:
60N to 60S and 0 to 360 Longitude
3000 rows x 9000 cols

first value is NW most pixel centered at .02 lon and 59.98 N lat.
next value is N most lat + .06 lon at .10 lon  and so on.
last value is SE corner.  

NOTES:  there are "realtime" data up near the current time.
	the realtime data begin to be reprocessed using more consistent
	input data after 2 days delay.  The reprocessing can require
	up to 24 hours.  

	Efforts are underway to clear out any leftover "realtime" data
	from this dataset.
	
----------
12/17/09
Dan Braithwaite
[email removed]
Center for Hydrometeorology and Remote Sensing (CHRS)
Department of Civil and Environmen

In [57]:
%%bash
# [orig cell 56 | env: precip | status: ok]
B=https://persiann.eng.uci.edu/CHRSdata/PERSIANN-CCS/hrly
curl -s --max-time 120 $B/2020/ -o p20.html; wc -c p20.html; grep -o 'href="rgccs[^"]*"' p20.html | sed -n '1,3p'; grep -o 'href="rgccs[^"]*"' p20.html | wc -l
curl -s --max-time 120 -o pccs_test.bin.gz -w "%{http_code} %{size_download}\n" $B/2020/rgccs1h2014722.bin.gz

 2003504 p20.html
href="rgccs1h2000100.bin.gz"
href="rgccs1h2000101.bin.gz"
href="rgccs1h2000102.bin.gz"
    8784
200 1730277


In [58]:
# [orig cell 57 | env: precip | status: ok]
a=np.frombuffer(gzip.decompress(open('pccs_test.bin.gz','rb').read()),'>i2').reshape(3000,9000)
print(np.unique(a[a<0])[:5])
lat_p=59.98-0.04*np.arange(3000); lon_p=0.02+0.04*np.arange(9000)
i=np.argmin(abs(lat_p-25.8)); j=np.argmin(abs(lon_p-(360-80.3)))
print(a[i-2:i+3,j-2:j+3]/100)

[]
[[ 8.55  8.94  8.55 10.79  9.36]
 [10.96  8.91  8.52  8.91  8.78]
 [13.43  8.88  8.32  8.32  9.56]
 [ 8.28  7.89  7.53  8.28  8.35]
 [ 6.02  6.46  6.46  6.46  6.46]]


In [60]:
%%bash
# [orig cell 59 | env: precip | status: ok]
python prod/nasa_od.py merra2 2>&1 | tail -15

merra days 2112 with granule 2112
stations ok 72
2000 5.8 min
merra2 done; fails 0 hours with data 22744 of 22744 6.1 min


In [61]:
# [orig cell 60 | env: precip | status: ok]
u='https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NA.list.v04r01.csv'
r=requests.get(u,timeout=300); print(r.status_code,len(r.content))
open('ibtracs_NA.csv','wb').write(r.content)
ib=pd.read_csv('ibtracs_NA.csv',skiprows=[1],usecols=['SID','NAME','ISO_TIME','LAT','LON','USA_SSHS','NATURE','USA_WIND'],low_memory=False)
ib['t']=pd.to_datetime(ib.ISO_TIME); ib=ib[(ib.t>='2014-12-01')]
print(len(ib), ib.t.max(), ib.NAME.nunique())

200 57577752
12518 2026-10-11 12:00:00 132


In [62]:
# [orig cell 61 | env: precip | status: ok]
ib['LAT']=pd.to_numeric(ib.LAT,errors='coerce'); ib['LON']=pd.to_numeric(ib.LON,errors='coerce')
# interpolate each storm to hourly
tr=[]
for sid,g in ib.groupby('SID'):
    g=g.drop_duplicates('t').set_index('t').sort_index()
    hi=pd.date_range(g.index.min().ceil('h'),g.index.max().floor('h'),freq='h')
    gi=g[['LAT','LON']].reindex(g.index.union(hi)).interpolate('time').loc[hi]
    gi['SID']=sid; gi['NAME']=g.NAME.iloc[0]; gi['NATURE']=g.NATURE.reindex(hi,method='ffill').values; tr.append(gi)
TR=pd.concat(tr).rename_axis('t').reset_index()
def hav(a,b,c,d):
    p=np.pi/180; x=np.sin((c-a)*p/2)**2+np.cos(a*p)*np.cos(c*p)*np.sin((d-b)*p/2)**2; return 6371*2*np.arcsin(np.sqrt(x))
e20=pd.read_csv('out/events_ge20.csv',parse_dates=['peak_hour_utc','first_hour_utc','last_hour_utc'])
TRi=TR.set_index('t')
def tcinfo(row):
    t=row.peak_hour_utc+pd.Timedelta('30min')
    t=t.floor('h')
    if t not in TRi.index: return pd.Series(dict(tc_dist_km=np.nan,tc_name=None))
    g=TRi.loc[[t]]; d=hav(row.lat,row.lon,g.LAT.values,g.LON.values); k=d.argmin()
    return pd.Series(dict(tc_dist_km=d[k],tc_name=g.NAME.values[k]))
tc=e20.apply(tcinfo,axis=1); e20=pd.concat([e20,tc],axis=1)
e20['is_TC']=e20.tc_dist_km<=500
print(e20.is_TC.mean().round(3), e20[e20.is_TC].tc_name.value_counts().head(12).to_dict())
for T in [30,40,50]: print(T, e20[e20.peak_mm_h>=T].is_TC.mean().round(3))
TR.to_parquet('out/ibtracs_hourly.parquet')

0.057 {'IRMA': 53, 'MILTON': 27, 'SALLY': 26, 'IAN': 24, 'ETA': 22, 'DEBBY': 21, 'ALEX': 19, 'EMILY': 18, 'COLIN': 15, 'ARLENE': 14, 'JULIA': 12, 'UNNAMED': 12}
30 0.062
40 0.062
50 0.07


In [63]:
%%bash
# [orig cell 62 | env: precip | status: ok]
sed -i.bak "s#ce='/PRECTOTCORR\[0:1:23\]\[226:1:246\]\[143:1:163\]'; lat=np.arange(226,247)\*0.5-90; lon=np.arange(143,164)\*0.625-180#ce='/PRECTOTCORR[0:1:23][220:1:252][136:1:172]'; lat=np.arange(220,253)*0.5-90; lon=np.arange(136,173)*0.625-180#" prod/nasa_od.py && grep -c "220:1:252" prod/nasa_od.py && python prod/nasa_od.py merra2 2>&1 | tail -4

1
merra days 2112 with granule 2112
stations ok 79
2000 8.1 min
merra2 done; fails 0 hours with data 22744 of 22744 11.6 min
